# Residual Multimodal Transformer stability

This experiment converts the visual-first residual GRU fusion model into a
compact multimodal Transformer while preserving the same residual-fusion logic,
fixed subject split, ten-seed stability evaluation, and missing-visual
robustness experiments.

The visual Transformer produces the primary prediction. Compact motion and
heart-rate Transformers can only add a bounded, gated residual correction that
is initialized to zero. This retains a visual-only fallback while evaluating
whether modality-scaled Transformer encoders can exploit complementary sensor
information.


In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

from tqdm import tqdm
import json
import time
import os

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [2]:
df = pd.read_parquet("Data/dipser_dataset.parquet", engine="pyarrow")
df["age_group"] = pd.cut(df["age"], bins=[13, 20, 22, 26, 44]) # bins=[13, 17, 20, 22, 26, 44]
df.head()

,group,time,time_sec,image_path,metadata,subject,experiment,self_labeling emotion,self_labeling attention,labeler_02 attention,labeler_04 attention,labeler_02 emotion,labeler_01 attention,labeler_03 emotion,labeler_03 attention,labeler_01 emotion,labeler_04 emotion,self_labeling emotionfilled,self_labeling attentionfilled,labeler_02 attentionfilled,labeler_04 attentionfilled,labeler_02 emotionfilled,labeler_01 attentionfilled,labeler_03 emotionfilled,labeler_03 attentionfilled,labeler_01 emotionfilled,labeler_04 emotionfilled,samsung_rotation_vector value0_mean,samsung_rotation_vector value0_std,samsung_rotation_vector value1_mean,samsung_rotation_vector value1_std,samsung_rotation_vector value2_mean,samsung_rotation_vector value2_std,samsung_rotation_vector value3_mean,samsung_rotation_vector value3_std,samsung_rotation_vector value4_mean,samsung_rotation_vector value4_std,lsm6dso_gyroscope value0_mean,lsm6dso_gyroscope value0_std,lsm6dso_gyroscope value1_mean,lsm6dso_gyroscope value1_std,lsm6dso_gyroscope value2_mean,lsm6dso_gyroscope value2_std,samsung_linear_acceleration_sensor value0_mean,samsung_linear_acceleration_sensor value0_std,samsung_linear_acceleration_sensor value1_mean,samsung_linear_acceleration_sensor value1_std,samsung_linear_acceleration_sensor value2_mean,samsung_linear_acceleration_sensor value2_std,opt3007_light value0_mean,opt3007_light value0_std,heart_rate,heart_rate_std,accel_magnitude_mean,accel_magnitude_std,gyro_magnitude_mean,gyro_magnitude_std,labeler_05 emotion,labeler_05 attention,labeler_05 emotionfilled,labeler_05 attentionfilled,invalid_reason,gender,age,race,subject_experiment_id,attention,subject_id,age_group
0,group01,2026-05-08 10:40:43.047895,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_047895.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_047895.json,subject_01,experiment01,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109524,0.000163,-0.659717,0.000588,0.632875,0.000495,0.390186,0.000218,246.0,0.0,-0.006022,0.006482,-0.003838,0.003665,-0.000864,0.006845,0.052984,0.034713,-0.132399,0.047077,0.017549,0.043508,63.0,0.0,78.0,NaN,0.155556,0.041540,0.011095,0.005498,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
1,group01,2026-05-08 10:40:43.195317,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_195317.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_195317.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109485,0.000109,-0.660015,0.000520,0.632620,0.000426,0.390106,0.000225,246.0,0.0,-0.004508,0.005235,-0.004068,0.003267,-0.001026,0.005944,0.053056,0.036979,-0.124020,0.045047,0.012282,0.043478,63.0,0.0,78.0,NaN,0.148634,0.038779,0.009572,0.004378,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
2,group01,2026-05-08 10:40:43.295405,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_295405.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_295405.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109478,0.000101,-0.660194,0.000514,0.632473,0.000406,0.390043,0.000252,246.0,0.0,-0.004581,0.005193,-0.003897,0.003153,-0.000525,0.005644,0.054420,0.036670,-0.129478,0.044805,0.015730,0.040968,63.0,0.0,78.0,NaN,0.153129,0.038995,0.009384,0.004079,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
3,group01,2026-05-08 10:40:43.395835,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_395835.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_395835.json,subject_01,experime

In [3]:
df.shape

(889685, 69)

In [4]:
SEED = 42
SEQUENCE_LENGTH = 10
MAX_MISSING_VISUAL_FRAMES = SEQUENCE_LENGTH -1  # remove a sequence only when all the images are unavailable

# Label-imbalance intervention toggles.
USE_WEIGHTED_LOSS = True
USE_WEIGHTED_SAMPLER = False
WEIGHT_ALPHA = 0.5       # 0=no weighting, 1=full inverse-frequency weighting
STRATIFY_COLUMN = 'age_group'
BATCH_SIZE = 32
NUM_WORKERS = 8

# ATTENTION_BINS = [2, 2.5, 3, 3.5, 4, 4.75]
ATTENTION_BINS = [2, 2.5, 3, 3.5, 4.75]

In [5]:
df[['attention', 'image_path', 'age', 'gender']].isna().sum()

attention     0
image_path    0
age           0
gender        0
dtype: int64

In [6]:
sensor_cols = [
    col for col in df.columns
    if (
       # "sensor" in col.lower()
        "acceleration" in col.lower()
        or "gyro" in col.lower()
      #  or "rotation" in col.lower()
        or "heart" in col.lower()
      #  or "light" in col.lower()
        or "accel" in col.lower()
    )
    and "std" not in col.lower()
]

# Heart rate is physiologically different from motion/device sensors, so Fusion
# follows Temporal Sensor and gives it a separate projection stream.
hr_cols = ["heart_rate"] if "heart_rate" in sensor_cols else []
motion_cols = [col for col in sensor_cols if col not in hr_cols]
hr_indices = [sensor_cols.index(col) for col in hr_cols]
motion_indices = [sensor_cols.index(col) for col in motion_cols]

len(sensor_cols), len(motion_cols), len(hr_cols), sensor_cols[:5]

(9,
 8,
 1,
 ['lsm6dso_gyroscope value0_mean',
  'lsm6dso_gyroscope value1_mean',
  'lsm6dso_gyroscope value2_mean',
  'samsung_linear_acceleration_sensor value0_mean',
  'samsung_linear_acceleration_sensor value1_mean'])

In [7]:
# Treat physiologically impossible HR values as missing before missing flags and scaling.
# The smartwatch can emit 0, which should not be interpreted as a real heart rate.
df.loc[df["heart_rate"] < 30, "heart_rate"] = np.nan

df[sensor_cols] = df[sensor_cols].astype(np.float32)

In [8]:
# keep only 1 frame per second (the first)
df_sec = (df.sort_values(["subject_experiment_id", "time_sec"])
      .groupby(["subject_experiment_id", "time_sec"])
      .first()
      .reset_index())
df_sec.shape

(111754, 69)

In [9]:
reconstructed_sequences = []

for sequence_id, sequence_df in df_sec.groupby("subject_experiment_id"):
    sequence_df = sequence_df.sort_values("time_sec")

    complete_seconds = pd.DataFrame({
        "time_sec": np.arange(
            sequence_df["time_sec"].min(),
            sequence_df["time_sec"].max() + 1
        )
    })

    reconstructed = complete_seconds.merge(
        sequence_df,
        on="time_sec",
        how="left"
    )

    reconstructed["subject_experiment_id"] = sequence_id
    reconstructed_sequences.append(reconstructed)

temporal_frame_dataset = pd.concat(reconstructed_sequences, ignore_index=True)

sequence_metadata = (
    df[["subject_experiment_id", "subject_id", "gender", "age", "age_group"]]
    .drop_duplicates("subject_experiment_id"))

# Restore metadata for reconstructed missing seconds. Sensor/image/target values
# stay missing unless observed, so missingness flags remain meaningful.
temporal_frame_dataset = temporal_frame_dataset.drop(
    columns=["subject_id", "gender", "age", "age_group"],
    errors="ignore").merge(sequence_metadata, on="subject_experiment_id", how="left")

temporal_frame_dataset["visual_missing"] = temporal_frame_dataset["image_path"].isna().astype(np.float32)
temporal_frame_dataset["motion_missing"] = temporal_frame_dataset[motion_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["hr_missing"] = temporal_frame_dataset[hr_cols].isna().all(axis=1).astype(np.float32) if hr_cols else 1.0
temporal_frame_dataset["sensor_missing"] = temporal_frame_dataset[sensor_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["sensor_partial_nan"] = (
    temporal_frame_dataset[sensor_cols].isna().any(axis=1)
    & ~temporal_frame_dataset[sensor_cols].isna().all(axis=1)
).astype(np.float32)

temporal_frame_dataset[["subject_experiment_id", "time_sec", "visual_missing", "motion_missing", "hr_missing", "sensor_missing", "attention"]].head()

,subject_experiment_id,time_sec,visual_missing,motion_missing,hr_missing,sensor_missing,attention
0,group01_experiment01_subject_01,0,0.0,0.0,0.0,0.0,3.25
1,group01_experiment01_subject_01,1,0.0,0.0,0.0,0.0,3.00
2,group01_experiment01_subject_01,2,0.0,0.0,0.0,0.0,3.00
3,group01_experiment01_subject_01,3,0.0,0.0,0.0,0.0,3.25
4,group01_experiment01_subject_01,4,0.0,0.0,0.0,0.0,3.25


In [10]:
temporal_frame_dataset.shape

(121631, 74)

In [11]:
# feature_index = pd.read_parquet("Data/resnet50_features/resnet50_frame_index.parquet")
# feature_store_path = "Data/resnet50_features/resnet50_frame_features.npy"
# VISUAL_FEATURE_DIM = 2048

# feature_index = pd.read_parquet('Data/All_clip_vitl14_features/All_clip_vitl14_frame_index.parquet')
# feature_store_path =  'Data/All_clip_vitl14_features/All_clip_vitl14_frame_features.npy'
# VISUAL_FEATURE_DIM = 768

feature_index = pd.read_parquet('Data/clip_vitl14_features/clip_vitl14_frame_index.parquet')
feature_store_path =  'Data/clip_vitl14_features/clip_vitl14_frame_features.npy'
VISUAL_FEATURE_DIM = 768

feature_row_by_path = dict(zip(feature_index["image_path"], feature_index["feature_row"]))
FEATURES = feature_store_path.split('/')[-2]

temporal_frame_dataset["feature_row"] = (
    temporal_frame_dataset["image_path"]
    .map(feature_row_by_path)
    .fillna(-1)
    .astype(np.int64))

missing_feature_rows = (
    (temporal_frame_dataset["visual_missing"] == 0)
    & (temporal_frame_dataset["feature_row"] == -1)
).sum()
print("Non-missing image rows without cached features:", missing_feature_rows)

Non-missing image rows without cached features: 0


In [12]:
# Select one fixed age-stratified split using demographic metadata only.
# Model predictions, labels, and test performance are never used to choose it.
SPLIT_SEARCH_TRIALS = 1_000
MIN_MALE_VAL_SUBJECTS = 3
MIN_MALE_TEST_SUBJECTS = 3


def demographic_distance(split_df, full_df, column):
    categories = sorted(full_df[column].astype(str).unique())
    full_dist = full_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    split_dist = split_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    return float(np.abs(split_dist - full_dist).sum())


def find_constrained_age_stratified_split(subject_df, trials=SPLIT_SEARCH_TRIALS, seed=SEED):
    candidates = []
    all_age_groups = set(subject_df["age_group"].astype(str))

    for offset in range(trials):
        split_seed = seed + offset
        try:
            train_sub, temp_sub = train_test_split(
                subject_df,
                test_size=0.3,
                stratify=subject_df["age_group"],
                random_state=split_seed,
            )
            val_sub, test_sub = train_test_split(
                temp_sub,
                test_size=0.5,
                stratify=temp_sub["age_group"],
                random_state=split_seed,
            )
        except ValueError:
            continue

        val_males = int((val_sub["gender"].astype(str).str.lower() == "male").sum())
        test_males = int((test_sub["gender"].astype(str).str.lower() == "male").sum())
        val_has_all_ages = set(val_sub["age_group"].astype(str)) == all_age_groups
        test_has_all_ages = set(test_sub["age_group"].astype(str)) == all_age_groups

        representation_penalty = (
            max(0, MIN_MALE_VAL_SUBJECTS - val_males) * 100
            + max(0, MIN_MALE_TEST_SUBJECTS - test_males) * 100
            + (0 if val_has_all_ages else 100)
            + (0 if test_has_all_ages else 100)
        )
        balance_score = sum(
            demographic_distance(split, subject_df, column)
            for split in [train_sub, val_sub, test_sub]
            for column in ["age_group", "gender"]
        )
        candidates.append(
            (
                representation_penalty,
                balance_score,
                split_seed,
                train_sub.copy(),
                val_sub.copy(),
                test_sub.copy(),
            )
        )

    if not candidates:
        raise RuntimeError("Could not construct an age-stratified subject split.")

    best = min(candidates, key=lambda item: (item[0], item[1], item[2]))
    if best[0] > 0:
        print("WARNING: No split satisfied every requested representation constraint.")
    return best[3], best[4], best[5], best[2], best[0], best[1]


subject_df = (
    temporal_frame_dataset[["subject_id", "age_group", "gender"]]
    .dropna(subset=["subject_id", "age_group", "gender"])
    .drop_duplicates("subject_id")
    .copy()
)

train_sub, val_sub, test_sub, SPLIT_RANDOM_STATE, split_penalty, split_balance_score = (
    find_constrained_age_stratified_split(subject_df)
)

train_subjects = train_sub["subject_id"]
val_subjects = val_sub["subject_id"]
test_subjects = test_sub["subject_id"]

train_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(train_subjects)].copy()
val_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(val_subjects)].copy()
test_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(test_subjects)].copy()

# Train-only sensor normalization, identical to the original notebook.
sensor_means = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].mean()
sensor_stds = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].std()
sensor_stds = sensor_stds.replace(0, np.nan).fillna(1.0)
sensor_means = sensor_means.fillna(0.0)


def apply_sensor_scaling(frame_df):
    frame_df = frame_df.copy()
    scaled = ((frame_df[sensor_cols] - sensor_means) / sensor_stds).astype(np.float32)
    scaled = scaled.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scaled.loc[frame_df["sensor_missing"] == 1, :] = 0.0
    frame_df.loc[:, sensor_cols] = scaled.to_numpy(dtype=np.float32)
    return frame_df


train_frames = apply_sensor_scaling(train_frames)
val_frames = apply_sensor_scaling(val_frames)
test_frames = apply_sensor_scaling(test_frames)

MOTION_INPUT_DIM = len(motion_cols) + 1
HR_INPUT_DIM = len(hr_cols) + 1
SENSOR_INPUT_DIM = len(sensor_cols) + 1


def split_demographic_table(split_df, split_name):
    rows = []
    for attribute in ["age_group", "gender"]:
        for group, count in split_df[attribute].astype(str).value_counts().sort_index().items():
            rows.append({
                "split": split_name,
                "attribute": attribute,
                "group": group,
                "subjects": int(count),
                "proportion": float(count / len(split_df)),
            })
    return pd.DataFrame(rows)


split_demographics = pd.concat(
    [
        split_demographic_table(train_sub, "train"),
        split_demographic_table(val_sub, "validation"),
        split_demographic_table(test_sub, "test"),
    ],
    ignore_index=True,
)

print(f"Selected demographic-only split random state: {SPLIT_RANDOM_STATE}")
print(f"Constraint penalty: {split_penalty}; demographic balance score: {split_balance_score:.4f}")
display(split_demographics)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_frames), len(val_frames), len(test_frames)],
    "subjects": [train_frames.subject_id.nunique(), val_frames.subject_id.nunique(), test_frames.subject_id.nunique()],
    "target_mean": [train_frames.attention.mean(), val_frames.attention.mean(), test_frames.attention.mean()],
    "visual_missing_rate": [train_frames.visual_missing.mean(), val_frames.visual_missing.mean(), test_frames.visual_missing.mean()],
    "motion_missing_rate": [train_frames.motion_missing.mean(), val_frames.motion_missing.mean(), test_frames.motion_missing.mean()],
    "hr_missing_rate": [train_frames.hr_missing.mean(), val_frames.hr_missing.mean(), test_frames.hr_missing.mean()],
    "sensor_missing_rate": [train_frames.sensor_missing.mean(), val_frames.sensor_missing.mean(), test_frames.sensor_missing.mean()],
})


Selected demographic-only split random state: 45
Constraint penalty: 0; demographic balance score: 0.3941


,split,attribute,group,subjects,proportion
0,train,age_group,"(13, 20]",13,0.333333
1,train,age_group,"(20, 22]",10,0.256410
2,train,age_group,"(22, 26]",10,0.256410
3,train,age_group,"(26, 44]",6,0.153846
4,train,gender,female,27,0.692308
5,train,gender,male,12,0.307692
6,validation,age_group,"(13, 20]",3,0.333333
7,validation,age_group,"(20, 22]",3,0.333333
8,validation,age_group,"(22, 26]",2,0.222222
9,validation,age_group,"(26, 44]",1,0.111111


,split,rows,subjects,target_mean,visual_missing_rate,motion_missing_rate,hr_missing_rate,sensor_missing_rate
0,train,84989,39,2.968285,0.086705,0.086705,0.095083,0.086705
1,val,19092,9,2.944667,0.070920,0.070920,0.074324,0.070920
2,test,17550,9,2.879391,0.065755,0.065755,0.114701,0.065755


In [13]:
def create_temporal_sequences(frame_df, sequence_length=SEQUENCE_LENGTH):
    sequences = []

    for sequence_id, sequence_df in frame_df.groupby("subject_experiment_id"):
        sequence_df = sequence_df.sort_values("time_sec").reset_index(drop=True)

        if len(sequence_df) < sequence_length:
            continue

        for i in range(sequence_length - 1, len(sequence_df)):
            target = sequence_df.iloc[i]["attention"]

            if pd.isna(target):
                continue

            history = sequence_df.iloc[i - sequence_length + 1:i + 1]
            visual_missing_flags = history["visual_missing"].astype(np.float32).values

            if visual_missing_flags.sum() > MAX_MISSING_VISUAL_FRAMES:
                continue

            motion_missing_flags = history["motion_missing"].astype(np.float32).values
            hr_missing_flags = history["hr_missing"].astype(np.float32).values
            sensor_missing_flags = history["sensor_missing"].astype(np.float32).values

            sequence_record = {
                "subject_experiment_id": sequence_id,
                "subject_id": sequence_df.iloc[i]["subject_id"],
                "gender": sequence_df.iloc[i]["gender"],
                "age": sequence_df.iloc[i]["age"],
                "age_group": sequence_df.iloc[i]["age_group"],
                "time_sec": int(sequence_df.iloc[i]["time_sec"]),
                "feature_rows": history["feature_row"].tolist(),
                "visual_missing_flags": visual_missing_flags.tolist(),
                "motion_missing_flags": motion_missing_flags.tolist(),
                "hr_missing_flags": hr_missing_flags.tolist(),
                "sensor_missing_flags": sensor_missing_flags.tolist(),
                "target": float(target),
            }

            # Store every scaled sensor as its own temporal column. Each value is
            # a length-SEQUENCE_LENGTH list, e.g. train_df["heart_rate"].iloc[0].
            for sensor_col in sensor_cols:
                sequence_record[sensor_col] = (
                    history[sensor_col]
                    .astype(np.float32)
                    .to_numpy(dtype=np.float32)
                    .tolist()
                )

            sequences.append(sequence_record)

    return pd.DataFrame(sequences)

In [14]:
train_df = create_temporal_sequences(train_frames)
val_df = create_temporal_sequences(val_frames)
test_df = create_temporal_sequences(test_frames)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
})

,split,sequences,subjects,target_mean
0,train,75189,39,2.971392
1,val,17208,9,2.948236
2,test,15894,9,2.881307


### Train Test Split

In [15]:
class MultimodalFusionDataset(Dataset):

    def __init__(self, sequence_df, feature_store_path):
        self.df = sequence_df.reset_index(drop=True)
        self.feature_store_path = feature_store_path
        self.feature_store = None

    def __len__(self):
        return len(self.df)

    def _features(self):
        if self.feature_store is None:
            self.feature_store = np.load(self.feature_store_path, mmap_mode="r")
        return self.feature_store

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        feature_rows = np.asarray(row["feature_rows"], dtype=np.int64)
        visual_features = np.zeros((len(feature_rows), VISUAL_FEATURE_DIM), dtype=np.float32)

        valid = feature_rows >= 0
        visual_features[valid] = self._features()[feature_rows[valid]]

        # Reconstruct the T x sensor_dim tensor from separate temporal sensor
        # columns instead of reading one packed sensor_values column.
        sensors = np.stack(
            [
                np.asarray(row[sensor_col], dtype=np.float32)
                for sensor_col in sensor_cols
            ],
            axis=1
        )
        sensors = torch.tensor(sensors, dtype=torch.float32)
        sensors = torch.nan_to_num(sensors, nan=0.0, posinf=0.0, neginf=0.0)

        motion = sensors[:, motion_indices]
        motion_missing = torch.tensor(row["motion_missing_flags"], dtype=torch.float32).unsqueeze(-1)
        motion = torch.cat([motion, motion_missing], dim=-1)

        if hr_indices:
            heart_rate = sensors[:, hr_indices]
        else:
            heart_rate = torch.zeros((sensors.shape[0], 0), dtype=torch.float32)
        hr_missing = torch.tensor(row["hr_missing_flags"], dtype=torch.float32).unsqueeze(-1)
        heart_rate = torch.cat([heart_rate, hr_missing], dim=-1)

        visual_features = torch.tensor(visual_features, dtype=torch.float32)
        visual_missing_flags = torch.tensor(row["visual_missing_flags"], dtype=torch.float32)
        target = torch.tensor(row["target"], dtype=torch.float32)
        sample_weight = torch.tensor(row["sample_weight"], dtype=torch.float32)

        return visual_features, motion, heart_rate, visual_missing_flags, target, sample_weight, idx

In [16]:
# Quantify target imbalance and create inverse-frequency train weights.
# Weights are learned from train only; val/test weights are diagnostic only.
def add_attention_bin_weights(alpha, train_df, val_df, test_df):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_bins = pd.cut(train_df["target"], bins=ATTENTION_BINS, include_lowest=True)
    bin_counts = train_bins.value_counts().sort_index()
    nonzero_counts = bin_counts[bin_counts > 0]

    bin_weights = (len(train_df) / (len(nonzero_counts) * nonzero_counts)) ** alpha

    for split_df in [train_df, val_df, test_df]:
        split_bins = pd.cut(split_df["target"], bins=ATTENTION_BINS, include_lowest=True)
        split_df["attention_bin"] = split_bins.astype(str)
        split_df["sample_weight"] = split_bins.map(bin_weights).astype(float).fillna(1.0)

    train_weight_mean = train_df["sample_weight"].mean()

    for split_df in [train_df, val_df, test_df]:
        split_df["sample_weight"] = split_df["sample_weight"] / train_weight_mean

    normalized_bin_weights = bin_weights / train_weight_mean

    imbalance_table = pd.DataFrame({
        "bin": bin_counts.index.astype(str),
        "train_count": bin_counts.values,
        "weight": [
            float(normalized_bin_weights.get(idx, np.nan))
            for idx in bin_counts.index
        ],
    })

    return train_df, val_df, test_df, imbalance_table

train_df, val_df, test_df, imbalance_table = add_attention_bin_weights(WEIGHT_ALPHA, train_df, val_df, test_df)
display(imbalance_table)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
    "mean_sample_weight": [train_df.sample_weight.mean(), val_df.sample_weight.mean(), test_df.sample_weight.mean()],
})

,bin,train_count,weight
0,"(1.999, 2.5]",15291,1.160744
1,"(2.5, 3.0]",35590,0.760835
2,"(3.0, 3.5]",18734,1.048671
3,"(3.5, 4.75]",5574,1.922521


,split,sequences,subjects,target_mean,mean_sample_weight
0,train,75189,39,2.971392,1.000000
1,val,17208,9,2.948236,0.981307
2,test,15894,9,2.881307,0.971843


In [17]:
train_df.shape, val_df.shape, test_df.shape

((75189, 23), (17208, 23), (15894, 23))

In [18]:
len(set(train_df['subject_experiment_id'])), len(set(val_df['subject_experiment_id'])) ,len(set(test_df['subject_experiment_id']))

(285, 64, 59)

In [19]:
train_dataset = MultimodalFusionDataset(train_df, feature_store_path)
val_dataset = MultimodalFusionDataset(val_df, feature_store_path)
test_dataset = MultimodalFusionDataset(test_df, feature_store_path)

if USE_WEIGHTED_SAMPLER:
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(train_df["sample_weight"].values, dtype=torch.double),
        num_samples=len(train_df),
        replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

### Training

In [20]:
class ResidualMultimodalFusionTransformer(nn.Module):

    def __init__(
        self,
        motion_input_dim,
        hr_input_dim,
        max_seq_len=SEQUENCE_LENGTH,
        visual_feature_dim=VISUAL_FEATURE_DIM,
        visual_dim=128,
        motion_dim=32,
        hr_dim=16,
        dropout=0.3,
        max_sensor_correction=0.5,
    ):
        super().__init__()
        self.max_seq_len = max_seq_len

        self.visual_projection = nn.Sequential(
            nn.Linear(visual_feature_dim + 1, visual_dim),
            nn.LayerNorm(visual_dim),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.motion_projection = nn.Sequential(
            nn.Linear(motion_input_dim, motion_dim),
            nn.LayerNorm(motion_dim),
            nn.GELU(),
        )
        self.hr_projection = nn.Sequential(
            nn.Linear(hr_input_dim, hr_dim),
            nn.LayerNorm(hr_dim),
            nn.GELU(),
        )

        # Each modality receives capacity proportional to its input complexity.
        self.visual_position = nn.Parameter(torch.zeros(max_seq_len, visual_dim))
        self.motion_position = nn.Parameter(torch.zeros(max_seq_len, motion_dim))
        self.hr_position = nn.Parameter(torch.zeros(max_seq_len, hr_dim))

        visual_layer = nn.TransformerEncoderLayer(
            d_model=visual_dim,
            nhead=4,
            dim_feedforward=256,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        motion_layer = nn.TransformerEncoderLayer(
            d_model=motion_dim,
            nhead=4,
            dim_feedforward=64,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        hr_layer = nn.TransformerEncoderLayer(
            d_model=hr_dim,
            nhead=2,
            dim_feedforward=32,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.visual_encoder = nn.TransformerEncoder(visual_layer, num_layers=2)
        self.motion_encoder = nn.TransformerEncoder(motion_layer, num_layers=1)
        self.hr_encoder = nn.TransformerEncoder(hr_layer, num_layers=1)
        self.visual_norm = nn.LayerNorm(visual_dim)
        self.motion_norm = nn.LayerNorm(motion_dim)
        self.hr_norm = nn.LayerNorm(hr_dim)

        self.visual_regressor = nn.Sequential(
            nn.Linear(visual_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

        fusion_dim = visual_dim + motion_dim + hr_dim
        self.sensor_dropout = nn.Dropout(0.25)
        self.residual_gate = nn.Sequential(
            nn.Linear(fusion_dim, 32),
            nn.GELU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )
        self.sensor_residual = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
        self.max_sensor_correction = max_sensor_correction

        # Begin as a visual-only Transformer. Sensor influence must be learned.
        nn.init.zeros_(self.sensor_residual[-1].weight)
        nn.init.zeros_(self.sensor_residual[-1].bias)
        nn.init.constant_(self.residual_gate[-2].bias, -2.0)

    @staticmethod
    def causal_mask(sequence_length, device):
        positions = torch.arange(sequence_length, device=device)
        return positions.unsqueeze(0) > positions.unsqueeze(1)

    def forward(self, visual_features, motion, heart_rate, visual_missing_flags):
        _, sequence_length, _ = visual_features.shape
        mask = self.causal_mask(sequence_length, visual_features.device)

        visual_missing_flags = visual_missing_flags.unsqueeze(-1)
        visual_input = torch.cat([visual_features, visual_missing_flags], dim=-1)
        visual_tokens = (
            self.visual_projection(visual_input)
            + self.visual_position[:sequence_length].unsqueeze(0)
        )
        motion_tokens = (
            self.motion_projection(motion)
            + self.motion_position[:sequence_length].unsqueeze(0)
        )
        hr_tokens = (
            self.hr_projection(heart_rate)
            + self.hr_position[:sequence_length].unsqueeze(0)
        )

        visual_encoded = self.visual_encoder(visual_tokens, mask=mask)
        motion_encoded = self.motion_encoder(motion_tokens, mask=mask)
        hr_encoded = self.hr_encoder(hr_tokens, mask=mask)

        visual_summary = self.visual_norm(visual_encoded[:, -1, :])
        motion_summary = self.motion_norm(motion_encoded[:, -1, :])
        hr_summary = self.hr_norm(hr_encoded[:, -1, :])
        visual_prediction = self.visual_regressor(visual_summary).squeeze(1)

        fusion_context = torch.cat(
            [visual_summary, motion_summary, hr_summary],
            dim=1,
        )
        fusion_context = self.sensor_dropout(fusion_context)
        gate = self.residual_gate(fusion_context).squeeze(1)
        residual = torch.tanh(self.sensor_residual(fusion_context).squeeze(1))
        return visual_prediction + self.max_sensor_correction * gate * residual


### Fixed-split multi-seed residual Transformer stability

This notebook measures whether visual-first residual fusion improves over the
visual-only baseline without inheriting the instability of the original large
Transformer fusion model.

The fixed split, weighted task loss, batch size, preprocessing, and ten training
seeds remain comparable to the baseline stability notebooks. The intentional
changes are the compact residual architecture, shuffled training batches, MSE
objective, RMSE checkpoint selection, and stronger weight decay.


In [21]:
import copy
import random
from importlib import reload
import src.evaluation as ev

ev = reload(ev)


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_model():
    return ResidualMultimodalFusionTransformer(
        motion_input_dim=MOTION_INPUT_DIM,
        hr_input_dim=HR_INPUT_DIM,
        max_seq_len=SEQUENCE_LENGTH,
        visual_feature_dim=VISUAL_FEATURE_DIM,
        visual_dim=128,
        motion_dim=32,
        hr_dim=16,
        dropout=0.3,
        max_sensor_correction=0.5,
    )


def make_train_loader():
    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=TRAIN_SHUFFLE,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=4,
    )


def weighted_task_loss(per_sample_loss, sample_weights):
    if USE_WEIGHTED_LOSS:
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)
    return per_sample_loss.mean()


def train_one_epoch_baseline(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    preds_all, labels_all = [], []

    for visual_features, motion, heart_rate, visual_missing_flags, labels, sample_weights, _idx in tqdm(loader, desc="Training", leave=False):
        visual_features = visual_features.to(device)
        motion = motion.to(device)
        heart_rate = heart_rate.to(device)
        visual_missing_flags = visual_missing_flags.to(device)
        labels = labels.to(device)
        sample_weights = sample_weights.to(device)

        preds = model(visual_features, motion, heart_rate, visual_missing_flags)
        loss = weighted_task_loss(criterion(preds, labels), sample_weights)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += float(loss.item())
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader),
        "mae": float(mean_absolute_error(labels_all, preds_all)),
        "rmse": float(np.sqrt(mean_squared_error(labels_all, preds_all))),
    }


def evaluate_loader(model, loader, sequence_df):
    model.eval()
    preds_all, labels_all, indices_all = [], [], []

    with torch.no_grad():
        for visual_features, motion, heart_rate, visual_missing_flags, labels, _sample_weights, idx in tqdm(loader, desc="Evaluating", leave=False):
            preds = model(
                visual_features.to(device),
                motion.to(device),
                heart_rate.to(device),
                visual_missing_flags.to(device),
            )
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())
            indices_all.extend(idx.detach().cpu().numpy())

    frame = sequence_df.iloc[np.asarray(indices_all, dtype=int)][
        ["subject_experiment_id", "subject_id", "time_sec", "attention_bin", "gender", "age", "age_group"]
    ].reset_index(drop=True)
    frame.insert(0, "true", np.asarray(labels_all, dtype=float))
    frame.insert(0, "pred", np.asarray(preds_all, dtype=float))
    frame = ev.add_robustness_metadata(
        frame, sequence_df, visual_missing_col="visual_missing_flags"
    )

    overall = ev.compute_prediction_metrics(frame)
    age_mae, age_worst, age_gap = ev.compute_group_mae(frame, "age_group")
    gender_mae, gender_worst, gender_gap = ev.compute_group_mae(frame, "gender")
    metrics = {
        "mae": overall["mae"],
        "rmse": overall["rmse"],
        "r2": overall["r2"],
        "true_mean": overall["true_mean"],
        "pred_mean": overall["pred_mean"],
        "age_worst_group_mae": age_worst,
        "age_gap": age_gap,
        "gender_worst_group_mae": gender_worst,
        "gender_gap": gender_gap,
        "age_mae_per_group": age_mae.to_dict(),
        "gender_mae_per_group": gender_mae.to_dict(),
    }
    return metrics, frame

In [22]:
class EarlyStopping:
    def __init__(self, patience, model_path):
        self.patience = patience
        self.model_path = model_path
        self.best_score = float("inf")
        self.best_epoch = None
        self.counter = 0

    def step(self, val_mae, model, epoch):
        if val_mae < self.best_score:
            self.best_score = float(val_mae)
            self.best_epoch = epoch
            self.counter = 0
            torch.save(copy.deepcopy(model.state_dict()), self.model_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


LOSS_TYPE = "mse"
criterion = nn.MSELoss(reduction="none")
EARLY_STOPPING_METRIC = "rmse"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 50
PATIENCE = 7
TRAIN_SHUFFLE = True

# Ten independent initialization/training seeds on the exact same subject split.
RUN_SEEDS = [42, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033]

RESULTS_DIR = "results/Multimodal Fusion Residual Transformer Stability"
MODEL_DIR = "models/Multimodal Fusion Residual Transformer Stability"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Fixed split random state: {SPLIT_RANDOM_STATE}")
print(f"Training shuffle: {TRAIN_SHUFFLE}")
print(f"Training seeds: {RUN_SEEDS}")

Fixed split random state: 45
Training shuffle: True
Training seeds: [42, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033]


In [23]:
run_records = []
test_predictions = {}

for run_seed in RUN_SEEDS:
    set_global_seed(run_seed)
    run_name = f"residual_transformer_fixed_age_split_seed{run_seed}"
    model_path = os.path.join(MODEL_DIR, f"{run_name}.pt")

    model = make_model().to(device)
    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.3, patience=3
    )
    early_stopping = EarlyStopping(PATIENCE, model_path)
    run_train_loader = make_train_loader()
    history = []

    print(f"\n=== {run_name} ===")
    for epoch in range(NUM_EPOCHS):
        train_metrics = train_one_epoch_baseline(model, run_train_loader, optimizer, criterion)
        val_metrics, _ = evaluate_loader(model, val_loader, val_df)
        monitor = val_metrics[EARLY_STOPPING_METRIC]
        scheduler.step(monitor)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_mae": train_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "val_rmse": val_metrics["rmse"],
            "val_r2": val_metrics["r2"],
        })
        print(
            f"Epoch {epoch + 1:02d} | train MAE {train_metrics['mae']:.4f} | "
            f"val MAE {val_metrics['mae']:.4f} | val R2 {val_metrics['r2']:.4f}"
        )
        if early_stopping.step(monitor, model, epoch):
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    val_metrics, val_predictions = evaluate_loader(model, val_loader, val_df)
    test_metrics, test_frame = evaluate_loader(model, test_loader, test_df)
    test_predictions[run_name] = test_frame

    val_prediction_path = os.path.join(RESULTS_DIR, f"{run_name}_val_predictions.csv")
    test_prediction_path = os.path.join(RESULTS_DIR, f"{run_name}_test_predictions.csv")
    ev.save_prediction_frame(val_predictions, val_prediction_path)
    ev.save_prediction_frame(test_frame, test_prediction_path)

    run_records.append({
        "run_name": run_name,
        "run_seed": run_seed,
        "best_epoch": early_stopping.best_epoch + 1,
        "num_epochs_run": len(history),
        "model_path": model_path,
        "val_prediction_path": val_prediction_path,
        "test_prediction_path": test_prediction_path,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "history": history,
    })

print("Finished all residual Transformer runs.")

/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



=== residual_transformer_fixed_age_split_seed42 ===


Epoch 01 | train MAE 0.3434 | val MAE 0.3611 | val R2 -0.2671


Epoch 02 | train MAE 0.2708 | val MAE 0.3266 | val R2 -0.0544


Epoch 03 | train MAE 0.2518 | val MAE 0.2833 | val R2 0.1941


...

Epoch 07 | train MAE 0.2197 | val MAE 0.2948 | val R2 0.0867


Epoch 08 | train MAE 0.2083 | val MAE 0.2996 | val R2 0.0429


Epoch 09 | train MAE 0.2049 | val MAE 0.3009 | val R2 0.0182


Epoch 10 | train MAE 0.2042 | val MAE 0.3038 | val R2 0.0338
Early stopping triggered


Finished all residual Transformer runs.


In [24]:
metric_rows = []
for run in run_records:
    row = {
        "run_name": run["run_name"],
        "run_seed": run["run_seed"],
        "best_epoch": run["best_epoch"],
        "num_epochs_run": run["num_epochs_run"],
    }
    row.update({f"val_{key}": value for key, value in run["val_metrics"].items() if not isinstance(value, dict)})
    row.update({f"test_{key}": value for key, value in run["test_metrics"].items() if not isinstance(value, dict)})
    metric_rows.append(row)

seed_results = pd.DataFrame(metric_rows)

summary_metrics = [
    "test_mae",
    "test_rmse",
    "test_r2",
    "test_age_worst_group_mae",
    "test_age_gap",
    "test_gender_worst_group_mae",
    "test_gender_gap",
]
seed_summary = (
    seed_results[summary_metrics]
    .agg(["mean", "std", "min", "median", "max"])
    .T
    .reset_index(names="metric")
)

display(seed_results.round(4))
display(seed_summary.round(4))


,run_name,run_seed,best_epoch,num_epochs_run,val_mae,val_rmse,val_r2,val_true_mean,val_pred_mean,val_age_worst_group_mae,val_age_gap,val_gender_worst_group_mae,val_gender_gap,test_mae,test_rmse,test_r2,test_true_mean,test_pred_mean,test_age_worst_group_mae,test_age_gap,test_gender_worst_group_mae,test_gender_gap
0,residual_transformer_fixed_age_split_seed42,42,3,10,0.2833,0.3776,0.1941,2.9482,2.9943,0.3676,0.1249,0.2960,0.0344,0.2923,0.3810,0.0807,2.8813,2.9291,0.3434,0.1498,0.2988,0.0193
1,residual_transformer_fixed_age_split_seed2025,2025,5,12,0.2952,0.3942,0.1218,2.9482,2.9466,0.3402,0.0741,0.2977,0.0069,0.2942,0.3823,0.0748,2.8813,2.8862,0.3711,0.1773,0.3065,0.0361
2,residual_transformer_fixed_age_split_seed2026,2026,4,11,0.2935,0.3863,0.1563,2.9482,2.9830,0.3877,0.1404,0.3072,0.0371,0.3118,0.3960,0.0073,2.8813,2.9248,0.3697,0.1573,0.3244,0.0371
3,residual_transformer_fixed_age_split_seed2027,2027,8,15,0.2884,0.3808,0.1803,2.9482,2.9549,0.3564,0.1136,0.2912,0.0078,0.3091,0.4016,-0.0214,2.8813,2.9151,0.3563,0.1420,0.3340,0.0733
4,residual_transformer_fixed_age_split_seed2028,2028,3,10,0.2921,0.3870,0.1532,2.9482,3.0172,0.3576,0.1107,0.2941,0.0054,0.2983,0.3760,0.1048,2.8813,2.9690,0.3352,0.1128,0.3122,0.0212
5,residual_transformer_fixed_age_split_seed2029,2029,10,17,0.2874,0.3853,0.1606,2.9482,2.9280,0.3235,0.0586,0.2930,0.0151,0.2838,0.3641,0.1606,2.8813,2.8606,0.3230,0.1217,0.2914,0.0225
6,residual_transformer_fixed_age_split_seed2030,2030,5,12,0.2886,0.3838,0.1673,2.9482,2.9839,0.3565,0.1052,0.2961,0.0205,0.2899,0.3736,0.1162,2.8813,2.9582,0.3366,0.1287,0.2965,0.0193
7,residual_transformer_fixed_age_split_seed2031,2031,8,15,0.2876,0.3841,0.1658,2.9482,2.9325,0.3366,0.0830,0.2985,0.0296,0.2897,0.3782,0.0945,2.8813,2.8836,0.3501,0.1455,0.2992,0.0280
8,residual_transformer_fixed_age_split_seed2032,2032,7,14,0.2948,0.3879,0.1493,2.9482,2.9525,0.3559,0.1157,0.2998,0.0136,0.2836,0.3692,0.1370,2.8813,2.9177,0.3287,0.1218,0.2928,0.0271
9,residual_transformer_fixed_age_split_seed2033,2033,3,10,0.2874,0.3897,0.1414,2.9482,2.9820,0.3685,0.1370,0.2977,0.0278,0.3096,0.3921,0.0267,2.8813,2.9715,0.3675,0.1748,0.3150,0.0159


,metric,mean,std,min,median,max
0,test_mae,0.2962,0.0106,0.2836,0.2932,0.3118
1,test_rmse,0.3814,0.0119,0.3641,0.3796,0.4016
2,test_r2,0.0781,0.0580,-0.0214,0.0876,0.1606
3,test_age_worst_group_mae,0.3482,0.0175,0.3230,0.3468,0.3711
4,test_age_gap,0.1432,0.0223,0.1128,0.1437,0.1773
5,test_gender_worst_group_mae,0.3071,0.0142,0.2914,0.3028,0.3340
6,test_gender_gap,0.0300,0.0168,0.0159,0.0248,0.0733


In [25]:
mean_baseline_predictions = ev.make_mean_baseline_predictions(train_df, test_df)
mean_baseline_metrics = ev.compute_prediction_metrics(mean_baseline_predictions)

# Diagnostic seed ensemble: average predictions from all independently trained
# models. This is not treated as a single-model baseline.
ensemble_frame = next(iter(test_predictions.values())).copy()
ensemble_frame["pred"] = np.mean(
    [frame["pred"].to_numpy(dtype=float) for frame in test_predictions.values()],
    axis=0,
)
ensemble_metrics = ev.compute_prediction_metrics(ensemble_frame)
_, ensemble_age_worst, ensemble_age_gap = ev.compute_group_mae(ensemble_frame, "age_group")
_, ensemble_gender_worst, ensemble_gender_gap = ev.compute_group_mae(ensemble_frame, "gender")
ensemble_metrics.update({
    "age_worst_group_mae": ensemble_age_worst,
    "age_gap": ensemble_age_gap,
    "gender_worst_group_mae": ensemble_gender_worst,
    "gender_gap": ensemble_gender_gap,
})

reference_table = pd.DataFrame([
    {"model": "train-mean constant baseline", **mean_baseline_metrics},
    {"model": "ten-seed prediction ensemble (diagnostic)", **ensemble_metrics},
])

display(reference_table.round(4))


,model,n_samples,mae,rmse,r2,tolerant_accuracy,tolerance,one_off_accuracy,binary_threshold,binary_accuracy,binary_f1,binary_roc_auc,binary_confusion_matrix,true_mean,pred_mean,age_worst_group_mae,age_gap,gender_worst_group_mae,gender_gap
0,train-mean constant baseline,15894,0.3224,0.4075,-0.0514,0.2191,0.15,1.0000,3.0,0.5470,0.0000,0.5000,"[[8694, 0], [7200, 0]]",2.8813,2.9714,NaN,NaN,NaN,NaN
1,ten-seed prediction ensemble (diagnostic),15894,0.2853,0.3677,0.1439,0.3418,0.15,0.9999,3.0,0.6487,0.5323,0.6838,"[[7134, 1560], [4023, 3177]]",2.8813,2.9216,0.3403,0.1454,0.2925,0.0213


In [26]:
BOOTSTRAP_RUNS = 1000
bootstrap_summary, _bootstrap_samples = ev.bootstrap_table(
    test_predictions,
    cluster_col="subject_id",
    n_boot=BOOTSTRAP_RUNS,
    seed=SEED,
)

# Quantify how much each test subject's MAE changes across training seeds.
subject_rows = []
for run_name, frame in test_predictions.items():
    per_subject = (
        frame.assign(abs_error=np.abs(frame["true"].astype(float) - frame["pred"].astype(float)))
        .groupby("subject_id", observed=True)
        .agg(
            samples=("abs_error", "size"),
            mae=("abs_error", "mean"),
            true_mean=("true", "mean"),
            pred_mean=("pred", "mean"),
        )
        .reset_index()
    )
    per_subject.insert(0, "run_name", run_name)
    subject_rows.append(per_subject)

subject_seed_metrics = pd.concat(subject_rows, ignore_index=True)
subject_stability = (
    subject_seed_metrics.groupby("subject_id", observed=True)
    .agg(
        samples=("samples", "first"),
        true_mean=("true_mean", "first"),
        mean_mae=("mae", "mean"),
        sd_mae=("mae", "std"),
        min_mae=("mae", "min"),
        max_mae=("mae", "max"),
    )
    .reset_index()
    .sort_values("mean_mae", ascending=False)
)

display(bootstrap_summary.round(4))
display(subject_stability.round(4))


,model,metric,mean,ci_low,ci_high
0,residual_transformer_fixed_age_split_seed42,mae,0.2929,0.2563,0.3445
1,residual_transformer_fixed_age_split_seed42,rmse,0.3809,0.3310,0.4429
2,residual_transformer_fixed_age_split_seed42,r2,0.0423,-0.2857,0.2393
3,residual_transformer_fixed_age_split_seed42,gender_gap,0.0413,0.0000,0.1255
4,residual_transformer_fixed_age_split_seed42,gender_worst_group_mae,0.3099,0.2658,0.3759
...,...,...,...,...,...
65,residual_transformer_fixed_age_split_seed2033,r2,-0.0045,-0.2661,0.1730
66,residual_transformer_fixed_age_split_seed2033,gender_gap,0.0416,0.0004,0.1366
67,residual_transformer_fixed_age_split_seed2033,gender_worst_group_mae,0.3258,0.2806,0.3663
68,residual_transformer_fixed_age_split_seed2033,age_gap,0.1532,0.0557,0.2662


,subject_id,samples,true_mean,mean_mae,sd_mae,min_mae,max_mae
2,group01_subject_15,1374,2.5231,0.4692,0.0355,0.4183,0.5458
7,group02_subject_20,2459,2.7932,0.3197,0.0273,0.2797,0.3674
8,group03_subject_11,2523,3.1625,0.3100,0.0393,0.2585,0.3687
5,group02_subject_10,1320,2.8854,0.2946,0.0196,0.2677,0.3253
6,group02_subject_16,2090,2.8524,0.2944,0.0244,0.2528,0.3411
1,group01_subject_10,1610,2.8425,0.2671,0.0406,0.2087,0.3137
4,group02_subject_07,1050,3.0031,0.2514,0.0271,0.2224,0.3230
3,group01_subject_18,1855,2.8776,0.2514,0.0207,0.2132,0.2806
0,group01_subject_08,1613,2.8786,0.2050,0.0099,0.1927,0.2224


In [27]:
# Paired robustness comparison against the completed visual baseline.
# Positive MAE gain means residual Transformer reduced error relative to visual-only.
ROBUSTNESS_SUBSETS = {
    "all_test_windows": None,
    "at_least_2_missing_images": "at_least_2_missing_images",
    "at_least_4_missing_images": "at_least_4_missing_images",
}
PAIRED_BOOTSTRAP_RUNS = 1000
VISUAL_RESULTS_DIR = "results/Temporal Visual Baseline Stability"


def paired_subject_bootstrap_mae_gain(
    baseline_frame,
    candidate_frame,
    subset_col=None,
    n_boot=PAIRED_BOOTSTRAP_RUNS,
    seed=SEED,
):
    keys = ["subject_experiment_id", "time_sec"]
    metadata_cols = ["subject_id"]
    if subset_col is not None:
        metadata_cols.append(subset_col)

    paired = baseline_frame[keys + metadata_cols + ["true", "pred"]].merge(
        candidate_frame[keys + ["pred"]],
        on=keys,
        how="inner",
        suffixes=("_visual", "_fusion"),
    )
    if subset_col is not None:
        paired = paired[paired[subset_col].fillna(False).astype(bool)]

    paired["mae_gain"] = (
        np.abs(paired["pred_visual"] - paired["true"])
        - np.abs(paired["pred_fusion"] - paired["true"])
    )
    subjects = paired["subject_id"].dropna().unique()
    if len(subjects) == 0:
        return {
            "n_samples": 0,
            "n_subjects": 0,
            "mae_gain": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    grouped = {
        subject: group
        for subject, group in paired.groupby("subject_id", sort=False)
    }
    rng = np.random.default_rng(seed)
    gains = []
    for _ in range(n_boot):
        sampled_subjects = rng.choice(subjects, size=len(subjects), replace=True)
        sampled = pd.concat(
            [grouped[subject] for subject in sampled_subjects],
            ignore_index=True,
        )
        gains.append(float(sampled["mae_gain"].mean()))

    return {
        "n_samples": int(len(paired)),
        "n_subjects": int(len(subjects)),
        "mae_gain": float(paired["mae_gain"].mean()),
        "ci_low": float(np.quantile(gains, 0.025)),
        "ci_high": float(np.quantile(gains, 0.975)),
    }


paired_robustness_rows = []
paired_bootstrap_rows = []

for run_seed in RUN_SEEDS:
    visual_name = f"visual_baseline_fixed_age_split_seed{run_seed}"
    fusion_name = f"residual_transformer_fixed_age_split_seed{run_seed}"
    visual_path = os.path.join(VISUAL_RESULTS_DIR, f"{visual_name}_test_predictions.csv")

    if not os.path.exists(visual_path):
        print(f"Skipping paired robustness comparison; missing {visual_path}")
        continue

    visual_frame = pd.read_csv(visual_path)
    visual_frame = ev.add_robustness_metadata(
        visual_frame,
        test_df,
        visual_missing_col="visual_missing_flags",
    )
    fusion_frame = test_predictions[fusion_name]

    comparison = ev.paired_robustness_comparison(
        {"visual": visual_frame, "residual_transformer": fusion_frame},
        baseline_model="visual",
        candidate_model="residual_transformer",
        subsets=ROBUSTNESS_SUBSETS,
    )
    comparison.insert(0, "run_seed", run_seed)
    paired_robustness_rows.append(comparison)

    for subset_name, subset_col in ROBUSTNESS_SUBSETS.items():
        result = paired_subject_bootstrap_mae_gain(
            visual_frame,
            fusion_frame,
            subset_col=subset_col,
            n_boot=PAIRED_BOOTSTRAP_RUNS,
            seed=SEED + run_seed,
        )
        paired_bootstrap_rows.append({
            "run_seed": run_seed,
            "subset": subset_name,
            **result,
        })

paired_robustness_summary = pd.concat(
    paired_robustness_rows,
    ignore_index=True,
) if paired_robustness_rows else pd.DataFrame()
paired_robustness_bootstrap = pd.DataFrame(paired_bootstrap_rows)

display(paired_robustness_summary.round(4))
display(paired_robustness_bootstrap.round(4))


,run_seed,baseline_model,candidate_model,subset,n_samples,baseline_mae,candidate_mae,candidate_mae_gain,candidate_better_rate
0,42,visual,residual_transformer,all_test_windows,15894,0.3088,0.2923,0.0166,0.5735
1,42,visual,residual_transformer,at_least_2_missing_images,1584,0.3214,0.3096,0.0118,0.5518
2,42,visual,residual_transformer,at_least_4_missing_images,176,0.3593,0.3520,0.0072,0.4773
3,2025,visual,residual_transformer,all_test_windows,15894,0.2915,0.2942,-0.0027,0.5079
4,2025,visual,residual_transformer,at_least_2_missing_images,1584,0.3140,0.3094,0.0046,0.5227
5,2025,visual,residual_transformer,at_least_4_missing_images,176,0.3393,0.3519,-0.0126,0.4886
6,2026,visual,residual_transformer,all_test_windows,15894,0.2930,0.3118,-0.0188,0.4377
7,2026,visual,residual_transformer,at_least_2_missing_images,1584,0.2943,0.3238,-0.0295,0.4141
8,2026,visual,residual_transformer,at_least_4_missing_images,176,0.3456,0.3471,-0.0016,0.4602
9,2027,visual,residual_transformer,all_test_windows,15894,0.2973,0.3091,-0.0119,0.4799


,run_seed,subset,n_samples,n_subjects,mae_gain,ci_low,ci_high
0,42,all_test_windows,15894,9,0.0166,-0.0073,0.0381
1,42,at_least_2_missing_images,1584,9,0.0118,-0.0173,0.0462
2,42,at_least_4_missing_images,176,9,0.0072,-0.0352,0.0672
3,2025,all_test_windows,15894,9,-0.0027,-0.0255,0.0281
4,2025,at_least_2_missing_images,1584,9,0.0046,-0.0279,0.0410
5,2025,at_least_4_missing_images,176,9,-0.0126,-0.0605,0.0626
6,2026,all_test_windows,15894,9,-0.0188,-0.0517,0.0162
7,2026,at_least_2_missing_images,1584,9,-0.0295,-0.0606,0.0062
8,2026,at_least_4_missing_images,176,9,-0.0016,-0.0336,0.0619
9,2027,all_test_windows,15894,9,-0.0119,-0.0314,0.0061


In [ ]:
# Synthetic visual-missingness stress test.
# Natural missing images remain missing. Additional available images are hidden
# deterministically until each eligible 10-second window reaches the requested
# total missing-image count.
import hashlib

SYNTHETIC_MASK_LEVELS = [4, 6, 8]
SYNTHETIC_MASK_SEED = 20260612
SYNTHETIC_BOOTSTRAP_RUNS = 1000


def make_synthetic_masked_sequences(
    sequence_df,
    visual_missing_col,
    target_missing,
    mask_seed=SYNTHETIC_MASK_SEED,
):
    masked_rows = []

    for _, row in sequence_df.iterrows():
        flags = np.asarray(row[visual_missing_col], dtype=np.float32).copy()
        feature_rows = np.asarray(row["feature_rows"], dtype=np.int64).copy()
        natural_missing = int(flags.sum())

        # Do not unmask naturally missing images. Windows already above the
        # requested level are excluded from that exact-level condition.
        if natural_missing > target_missing:
            continue

        additional_missing = target_missing - natural_missing
        candidates = np.flatnonzero((flags == 0) & (feature_rows >= 0))
        if len(candidates) < additional_missing:
            continue

        key = (
            f"{row['subject_experiment_id']}|{int(row['time_sec'])}|"
            f"{target_missing}|{mask_seed}"
        ).encode("utf-8")
        stable_seed = int.from_bytes(hashlib.sha256(key).digest()[:8], "little")
        rng = np.random.default_rng(stable_seed)
        chosen = rng.choice(candidates, size=additional_missing, replace=False)

        flags[chosen] = 1.0
        feature_rows[chosen] = -1

        updated = row.copy()
        updated[visual_missing_col] = flags.tolist()
        updated["feature_rows"] = feature_rows.tolist()
        updated["natural_visual_missing_count"] = natural_missing
        updated["synthetic_visual_missing_count"] = int(flags.sum())
        updated["synthetic_mask_level"] = target_missing
        masked_rows.append(updated)

    return pd.DataFrame(masked_rows).reset_index(drop=True)


def synthetic_seed_summary_table(seed_metrics):
    rows = []
    for level, group in seed_metrics.groupby("mask_level", observed=True):
        for metric in ["mae", "rmse", "r2"]:
            values = group[metric].astype(float)
            rows.append({
                "mask_level": int(level),
                "metric": metric,
                "mean": float(values.mean()),
                "std": float(values.std()),
                "min": float(values.min()),
                "max": float(values.max()),
            })
    return pd.DataFrame(rows)


synthetic_masking_predictions = {}
synthetic_masking_records = []
synthetic_masking_ensemble_rows = []
synthetic_masking_bootstrap_rows = []
paired_synthetic_rows = []
paired_synthetic_bootstrap_rows = []

fusion_loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "shuffle": False,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": NUM_WORKERS > 0,
    "prefetch_factor": 4 if NUM_WORKERS > 0 else None,
}
fusion_loader_kwargs = {
    key: value for key, value in fusion_loader_kwargs.items() if value is not None
}

for mask_level in SYNTHETIC_MASK_LEVELS:
    masked_test_df = make_synthetic_masked_sequences(
        test_df,
        visual_missing_col="visual_missing_flags",
        target_missing=mask_level,
    )
    masked_dataset = MultimodalFusionDataset(masked_test_df, feature_store_path)
    masked_loader = DataLoader(masked_dataset, **fusion_loader_kwargs)
    level_frames = []

    print(
        f"Synthetic missing={mask_level}: {len(masked_test_df)} windows, "
        f"{masked_test_df['subject_id'].nunique()} subjects"
    )

    for run_seed in RUN_SEEDS:
        run_name = f"residual_transformer_fixed_age_split_seed{run_seed}"
        model_path = os.path.join(MODEL_DIR, f"{run_name}.pt")
        model = make_model().to(device)
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))

        metrics, frame = evaluate_loader(model, masked_loader, masked_test_df)
        frame["synthetic_mask_level"] = mask_level
        frame["synthetic_mask_seed"] = SYNTHETIC_MASK_SEED
        prediction_name = f"{run_name}_synthetic_missing{mask_level}"
        synthetic_masking_predictions[prediction_name] = frame
        level_frames.append(frame)
        synthetic_masking_records.append({
            "run_name": run_name,
            "run_seed": run_seed,
            "mask_level": mask_level,
            "n_samples": len(frame),
            "n_subjects": frame["subject_id"].nunique(),
            **metrics,
        })
        ev.save_prediction_frame(
            frame,
            os.path.join(RESULTS_DIR, f"{prediction_name}_predictions.csv"),
        )

        visual_path = os.path.join(
            "results/Temporal Visual Baseline Stability",
            f"visual_baseline_fixed_age_split_seed{run_seed}_synthetic_missing{mask_level}_predictions.csv",
        )
        if os.path.exists(visual_path):
            visual_frame = pd.read_csv(visual_path)
            comparison = ev.paired_robustness_comparison(
                {"visual": visual_frame, "residual_transformer": frame},
                baseline_model="visual",
                candidate_model="residual_transformer",
                subsets={f"exactly_{mask_level}_missing_images": None},
            )
            comparison.insert(0, "run_seed", run_seed)
            comparison.insert(1, "mask_level", mask_level)
            paired_synthetic_rows.append(comparison)

            paired_bootstrap = paired_subject_bootstrap_mae_gain(
                visual_frame,
                frame,
                subset_col=None,
                n_boot=SYNTHETIC_BOOTSTRAP_RUNS,
                seed=SYNTHETIC_MASK_SEED + mask_level + run_seed,
            )
            paired_synthetic_bootstrap_rows.append({
                "run_seed": run_seed,
                "mask_level": mask_level,
                **paired_bootstrap,
            })

    ensemble = level_frames[0].copy()
    ensemble["pred"] = np.mean(
        [frame["pred"].to_numpy(dtype=float) for frame in level_frames],
        axis=0,
    )
    ensemble_metrics = ev.compute_prediction_metrics(ensemble)
    synthetic_masking_ensemble_rows.append({
        "mask_level": mask_level,
        "n_subjects": ensemble["subject_id"].nunique(),
        **ensemble_metrics,
    })
    ev.save_prediction_frame(
        ensemble,
        os.path.join(
            RESULTS_DIR,
            f"synthetic_missing{mask_level}_seed_ensemble_predictions.csv",
        ),
    )

    bootstrap_summary_level, _ = ev.cluster_bootstrap_ci(
        ensemble,
        cluster_col="subject_id",
        n_boot=SYNTHETIC_BOOTSTRAP_RUNS,
        seed=SYNTHETIC_MASK_SEED + mask_level,
    )
    bootstrap_summary_level.insert(0, "mask_level", mask_level)
    synthetic_masking_bootstrap_rows.append(bootstrap_summary_level.reset_index(names="metric"))

synthetic_masking_seed_metrics = pd.DataFrame(synthetic_masking_records)
synthetic_masking_seed_summary = synthetic_seed_summary_table(synthetic_masking_seed_metrics)
synthetic_masking_ensemble_metrics = pd.DataFrame(synthetic_masking_ensemble_rows)
synthetic_masking_ensemble_bootstrap = pd.concat(
    synthetic_masking_bootstrap_rows,
    ignore_index=True,
)
paired_synthetic_masking_summary = pd.concat(
    paired_synthetic_rows,
    ignore_index=True,
) if paired_synthetic_rows else pd.DataFrame()
paired_synthetic_masking_bootstrap = pd.DataFrame(paired_synthetic_bootstrap_rows)

display(synthetic_masking_seed_summary.round(4))
display(synthetic_masking_ensemble_metrics.round(4))
display(paired_synthetic_masking_summary.round(4))
display(paired_synthetic_masking_bootstrap.round(4))


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Synthetic missing=4: 15779 windows, 9 subjects


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/tra

In [ ]:
split_demographics.to_csv(os.path.join(RESULTS_DIR, "split_demographics.csv"), index=False)
seed_results.to_csv(os.path.join(RESULTS_DIR, "seed_results.csv"), index=False)
seed_summary.to_csv(os.path.join(RESULTS_DIR, "seed_summary.csv"), index=False)
reference_table.to_csv(os.path.join(RESULTS_DIR, "reference_baselines.csv"), index=False)
bootstrap_summary.to_csv(os.path.join(RESULTS_DIR, "subject_bootstrap_summary.csv"), index=False)
subject_seed_metrics.to_csv(os.path.join(RESULTS_DIR, "subject_seed_metrics.csv"), index=False)
subject_stability.to_csv(os.path.join(RESULTS_DIR, "subject_stability.csv"), index=False)
ev.save_prediction_frame(ensemble_frame, os.path.join(RESULTS_DIR, "seed_ensemble_test_predictions.csv"))
synthetic_masking_seed_metrics.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_seed_metrics.csv"), index=False)
synthetic_masking_seed_summary.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_seed_summary.csv"), index=False)
synthetic_masking_ensemble_metrics.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_ensemble_metrics.csv"), index=False)
synthetic_masking_ensemble_bootstrap.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_ensemble_bootstrap.csv"), index=False)

paired_synthetic_masking_summary.to_csv(os.path.join(RESULTS_DIR, "paired_visual_synthetic_masking_summary.csv"), index=False)
paired_synthetic_masking_bootstrap.to_csv(os.path.join(RESULTS_DIR, "paired_visual_synthetic_masking_bootstrap.csv"), index=False)
paired_robustness_summary.to_csv(os.path.join(RESULTS_DIR, "paired_visual_robustness_summary.csv"), index=False)
paired_robustness_bootstrap.to_csv(os.path.join(RESULTS_DIR, "paired_visual_robustness_bootstrap.csv"), index=False)
def json_safe(value):
    """Recursively convert pandas/NumPy objects and non-JSON dictionary keys."""
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value
    
with open(os.path.join(RESULTS_DIR, "run_manifest.json"), "w") as file:
    json.dump(
        json_safe({
            "experiment": "Residual multimodal Transformer fixed constrained split stability",
            "split_stratify": STRATIFY_COLUMN,
            "split_random_state": SPLIT_RANDOM_STATE,
            "split_constraint_penalty": split_penalty,
            "split_balance_score": split_balance_score,
            "run_seeds": RUN_SEEDS,
            "loss_type": LOSS_TYPE,
            "early_stopping_metric": EARLY_STOPPING_METRIC,
            "mean_baseline_metrics": mean_baseline_metrics,
            "ensemble_metrics": ensemble_metrics,
            "runs": run_records,
        }),
        file,
        indent=2,
        default=str,
    )
print(f"Saved residual Transformer stability results to: {RESULTS_DIR}")

### Reading the results

Use `seed_summary.csv` to judge whether the fixed split produces consistently
acceptable predictive performance. The mean, standard deviation, and worst seed
matter more than the single best seed.

The fixed split should only be used for the later fairness experiment if its
baseline consistently beats the train-mean constant predictor and its MAE/R2
remain acceptable across seeds.
